# Imports

In [1]:
from copy import deepcopy
import os
import json
import re
import requests
import tiktoken

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pdf.PineconePDFExtractor import PdfProcessor
from utils import split_docs, process_pdf

# Upserting to OJ DB
- Note: this section will skip execution if the DB is already registered

In [ ]:
url = 'localhost:4000'
endpoint = 'vector-db/upsert-with-source'
indexName, collectionName = 'benchmarking', 'constructive-dismissal3'

rsp = requests.post('/'.join(['http:/', url, 'vector-db/create-collection']), data={
    'indexName': indexName, 'collectionName': collectionName, 'lawType': 'Employment Law'})
print(rsp.status_code) 
print(rsp.text) # 201 on creation, 200 on alr registered

need_upsert = json.loads(rsp.text)["code"] == 201
print(need_upsert)

201
{"status":"created","code":201}
True


In [3]:
def upload(doc, index, namespace, filename):
    try:
        response = requests.post('/'.join(['http:/', url, endpoint]), data={
            'docs': doc, 
            'indexName':index, 
            'collectionName': namespace,
            'source':filename,
        })
        return response
    except Exception as e:
        print(e)
        return

In [4]:
if need_upsert:
    tiktoken.encoding_for_model('gpt-3.5-turbo')
    tokenizer = tiktoken.get_encoding('cl100k_base')

    # create the length function
    def tiktoken_len(text):
        tokens = tokenizer.encode(
            text,
            disallowed_special=()
        )
        return len(tokens)


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50,
        length_function=tiktoken_len,
        separators=["\n\n", "\n", " "]
    )

In [5]:
if need_upsert:
    lowercase_reg = r'^([a-z]|[0-9]{2,})'


    data = []
    data_dir = 'usecases/constructive_dismissal/caselaw'
    extractor = PdfProcessor(1)

    for fn in os.listdir(data_dir)[:10]:
        try:
            # extractor = PyPDFLoader(1)
            result = extractor.process_files([os.path.join(data_dir, fn)])
            pdf_text = result["documents"][0]["text"]
            # print(pdf_text)
            text = []
            paragraph = []
            for line in pdf_text.split('\n'):
                if 'Copyright' in line or 'page' in line.lower() or 'CanLII' in line or fn.split('.pdf')[0] in line:
                    pass
                else:
                    if re.match(lowercase_reg, line.strip()):
                        paragraph.append(line)
                    else:
                        text.append(' '.join(paragraph))
                        paragraph = [line]

            text.append(' '.join(paragraph))

        except Exception as e:
            print(f"Warning {os.path.join(data_dir, fn)}: PDF file is empty")
            continue

        data.append({'name': fn.split('.pdf')[0], 'content': re.sub(r'[\n]{3,}', '\n\n', '\n'.join([x.strip() for x in text])).strip()})

In [ ]:
if need_upsert:
    chunked_batch = []

    for x in data:
        print(x['name'] + '\n')
        # print(x['content'])

        chunked_texts = text_splitter.split_text(x['content'])
        if type(chunked_texts) is str:
            chunked_texts = [chunked_texts]
        # print(chunked_texts[0])
        x['court'] = ('court_of_appeals' if "COURT OF APPEAL" in x['content'] else 'supreme_court' if "SUPREME COURT" in x['content'] else 'lower_court')
        # chunked_batch.extend(chunked_texts)
        

        upload(chunked_texts, indexName, collectionName, x['name'])

Agostino v. Gary Bean Securities Ltd

Belton et al v Liberty Insurance Company of Canada

Blondeau v Holiday Ford Sales (1980) Ltd

Bowen v. Ritchie Bros Auctioneers Ltd

Brake v. PJ-M2R Restaurant Inc. 

Chapman v The Bank of Nova Scotia

Chapman v. GPM Investment Management

Chen v. Sigpro Wireless Inc

Ciciretto v Embassy Cleaners Inc

Conde v. Abrams Towing Services Ltd

{"statusCode":500,"error":"Error","message":"\n\n\nCollection second-benchmarking.second-benchmarking not registered. Call registerCollection first.","timestamp":"2025-11-08T04:39:14.952Z","path":"/vector-db/query","method":"POST"}


In [9]:
r = requests.post('/'.join(['http:/', url, 'vector-db/query']), data={
    'query': 'Amanda recently had a 50% wage cut after working for her company for five years.', 'indexName': indexName, 'collectionName': collectionName})
print(r.text) 

with open('sample.json', 'w') as f:
    eh = json.loads(r.text)
    json.dump(eh, f, indent=4)

[{"id":"b49cd7e0-85dd-4cdd-87e5-8bbc095e0ca0","version":5,"score":0.6092769,"payload":{"isMetadata":false,"text":"[44] After considering Ms. Brake's age, the length and nature of her employment, the manner  in which she had been dismissed, the low likelihood that she would ever again attain a simil ar  managerial position, and the impact on her of being unjustly dismissed in the context of her  character, reputation and circumstances, the trial judge found that a fair compensatory notice  period was 20 months, inclusive of any statutory severance requir ed by the Act.\n[45] He then awarded damages equivalent to her remuneration over a 20 -month time period,  without deduction for income she received during that period. The 20 months of compensation  amounted to $104,499.53. It was based on an annual salary of  $53,000, a $6,000 car  allowance, $1,307.76 for a cellphone and $2,391.84 in miscellaneous health benefits.","source":"Brake v. PJ-M2R Restaurant Inc. "},"vector":[-0.027123973,-

# Preprocessing the output `cd.jsonl`

In [19]:
data_dir = './usecases/constructive_dismissal/test'
caselaw_dir =  './usecases/constructive_dismissal/caselaw'
# out_file = 'cd_debug.jsonl'
out_file = 'cd2.jsonl'

In [ ]:
filenames = list(os.listdir(caselaw_dir))
filenames_short = [x.split()[0].lower() for x in filenames]

data = []
for dir in os.listdir(data_dir):
    print(data_dir, dir)
    supporting = split_docs(os.path.join(data_dir, dir)) 

    temp = dir.split()[0].lower()
    if temp in filenames_short:
        gold = process_pdf(caselaw_dir, filenames[filenames_short.index(temp)])
    
    data.append({'query': 'Has the employee been constructively dismissed?',
                 'supporting_docs': supporting,
                 'gold_label': '' 
                 })
print(data[0]['supporting_docs'])

./usecases/constructive_dismissal/test Brake v PJM2R Restaurant Inc
Benchmark Questions and Expected Answers.docx
Employee
Email confirming CSO and QSC Goals Met.docx
Email RE Conference.docx
Employee_Wrongful Dismissal Intake Form_ Termination Details_Brake.docx
GAP Document.docx
GAP Review June 27.docx
PR 00_07.docx
PR 2008.docx
PR 2009.docx
PR 2010.docx
PR 2011.docx
PR 2012.docx
SOR June 27.docx
Termination Letter.docx
Wendy's Corporate Documents.docx
Wrongful Dismissal Intake Form_ Compensation Details_Brake.docx
Wrongful Dismissal Intake Form_ Managerial Status_Brake.docx
[{'name': 'Email confirming CSO and QSC Goals Met', 'content': "Dear Jasmine,\n\nI am writing to confirm that you have successfully achieved the following key performance metrics at the Department Store location:\n\nA 0% Customer Service Opportunities (CSO) score\n\nA Quality, Service, and Cleanliness (QSC) score of 90% or higher\n\nI can also confirm that your GAP review was received later than scheduled; the 30

In [21]:
with open(out_file, 'w', encoding='utf-8') as f:
    for d in data:
        f.write(json.dumps(d) + '\n')